# 语音识别API使用手册

本手册将指导您如何从GitHub克隆项目、安装依赖并运行基于FastAPI的语音识别服务。

## 目录
1. [项目介绍](#项目介绍)
2. [环境准备](#环境准备)
3. [克隆项目](#克隆项目)
4. [安装依赖](#安装依赖)
5. [启动服务](#启动服务)
6. [API使用示例](#api使用示例)
7. [测试服务](#测试服务)

## 项目介绍

这是一个基于 FastAPI 构建的语音识别 API 服务，使用 Qwen3-ASR 模型进行音频转文字处理。Qwen3-ASR 是通义千问团队推出的自动语音识别模型，支持 52 种语言和方言的语音识别和语言识别。该模型基于 Qwen3-Omni 的强大音频理解能力和大规模语音训练数据，在开源 ASR 模型中达到了最先进的性能。

主要特点:

    🌍 支持 30 种语言和 22 种中文方言的语音识别
    ⚡ 0.6B 版本在保持高精度的同时实现极高吞吐量
    🎤 支持流式/离线统一推理
    📏 支持长音频转写

在本实验中，我们将：

    从 ModelScope 下载预转换的 OpenVINO 模型
    使用 OpenVINO 加载并运行语音识别
    体验交互式 Gradio 语音识别演示


## 主要功能
- 支持多种音频格式（WAV, MP3, FLAC, OGG等）
- 提供 RESTful API 接口
- 包含 Swagger UI 和 ReDoc 交互式文档
- 基于 OpenVINO 优化的推理引擎

## 环境准备

在开始之前，请确保您的系统已安装以下软件：
- Python 3.8 或更高版本
- pip 包管理器
- Git 版本控制工具

检查Python版本：

In [ ]:
!python --version

检查pip版本：

In [ ]:
!pip --version

检查Git是否安装：

In [ ]:
!git --version

## 克隆项目

使用Git从GitHub克隆项目到本地：

In [ ]:
# 克隆项目（请将下面的URL替换为实际的GitHub仓库地址）
!git clone https://github.com/ATA0018/speechRecognitionAPI.git
%cd speechRecognitionAPI

查看项目结构：

In [ ]:
!ls -la

## 安装依赖

创建虚拟环境（推荐）：

In [ ]:
# 创建虚拟环境
!python -m venv venv

# 激活虚拟环境（Linux/Mac）
!source venv/bin/activate

# 激活虚拟环境（Windows）
# !venv\Scripts\activate

安装项目依赖：

In [ ]:
!pip install -r requirements.txt

验证安装：

In [ ]:
# 检查关键依赖是否安装成功
import fastapi
import uvicorn
import numpy as np
print(f"FastAPI version: {fastapi.__version__}")
print("所有依赖安装成功！")

## 模型下载

In [ ]:
from pathlib import Path

# 直接指定模型下载路径到 static/models/Qwen3-ASR
model_dir = Path("static/models/Qwen3-ASR")

# 确保目录存在
model_dir.mkdir(parents=True, exist_ok=True)

if not any(model_dir.iterdir()):  # 检查目录是否为空
    print("🔄 开始下载模型到 static/models/Qwen3-ASR...")
    print(f"📍 下载路径: {model_dir.absolute()}")

    from modelscope import snapshot_download
    snapshot_download(
        "snake7gun/Qwen3-ASR-0.6B-fp16-ov",
        local_dir=str(model_dir)
    )
    print(f"✅ 模型已下载到: {model_dir}")
else:
    print(f"✅ 模型已存在: {model_dir}")
    print("⏭️  跳过下载步骤")


## 启动服务

在后台启动FastAPI服务：

In [ ]:
# 启动服务（在后台运行）
import subprocess
import time

# 启动uvicorn服务器
process = subprocess.Popen([
    "uvicorn", "main:app", 
    "--host", "0.0.0.0", 
    "--port", "8060", 
    "--reload"
])

# 等待服务启动
time.sleep(5)
print("服务已启动！访问 http://localhost:8060/docs 查看API文档")

或者直接在终端中运行：

In [ ]:
# 这种方式会在当前单元格中运行，会阻塞
# !uvicorn main:app --host 0.0.0.0 --port 8060 --reload

## API使用示例

### ---确认项目启动检查

In [ ]:
import requests

# 健康检查
response = requests.get("http://localhost:8060/")
print(f"状态码: {response.status_code}")
print(f"响应内容: {response.json()}")

### 语音识别API调用

首先准备一个测试音频文件：

In [ ]:
# 检查是否有测试音频文件
import os
if os.path.exists("data/sample_en.wav"):
    print("找到测试音频文件: data/sample_en.wav")
else:
    print("未找到测试音频文件，请准备一个WAV格式的音频文件")

调用语音识别API：

In [ ]:
# 语音识别请求示例
url = "http://localhost:8060/api/asr/transcribe"

# 如果有测试音频文件，可以使用它进行测试
if os.path.exists("data/sample_en.wav"):
    with open("data/sample_en.wav", "rb") as audio_file:
        files = {"file": ("sample_en.wav", audio_file, "audio/wav")}
        response = requests.post(url, files=files)
        
        print(f"状态码: {response.status_code}")
        if response.status_code == 200:
            result = response.json()
            print(f"识别结果: {result}")
        else:
            print(f"错误信息: {response.text}")
else:
    print("请提供一个音频文件进行测试")

### 使用curl命令测试

In [ ]:
# 使用curl命令测试API（如果存在测试音频文件）
if os.path.exists("data/sample_en.wav"):
    !curl -X POST "http://localhost:8060/api/asr/transcribe" \
      -H "accept: application/json" \
      -H "Content-Type: multipart/form-data" \
      -F "file=@data/sample_en.wav"
else:
    print("请提供一个音频文件进行测试")

## 测试服务

### 访问API文档

服务启动后，您可以通过浏览器访问以下地址：

- **Swagger UI**: http://localhost:8060/docs
- **ReDoc**: http://localhost:8060/redoc
- **OpenAPI JSON**: http://localhost:8060/openapi.json

这些交互式文档允许您直接在浏览器中测试API端点。

### 停止服务

In [ ]:
# 停止之前启动的服务进程
try:
    process.terminate()
    process.wait(timeout=5)
    print("服务已停止")
except:
    # 如果进程已经终止或不存在
    print("服务进程已终止或不存在")

## 常见问题

### 1. 端口被占用
如果8060端口被占用，可以更改端口号：
```bash
uvicorn main:app --host 0.0.0.0 --port 8080 --reload
```

### 2. 模型加载失败
确保模型文件存在于 `static/models/Qwen3-ASR/` 目录中。

### 3. 依赖安装问题
尝试更新pip后再安装依赖：
```bash
pip install --upgrade pip
pip install -r requirements.txt
```

## 总结

恭喜！您已经成功：
1. 从GitHub克隆了语音识别API项目
2. 安装了所有必要的依赖
3. 启动了FastAPI服务
4. 测试了API功能

现在您可以将此API集成到您的应用程序中，用于语音识别任务。祝你好运！